<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/Pre_mcmc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# ============================================================
# TIFA PAPER 3 — STAGE A
# Joint likelihood grid scan
#
# Datasets:
#   1. DESI DR1 BAO (w0, wa)
#   2. Pantheon+ compressed (d_L ratios)
#   3. Planck 2018 compressed (Omega_m, H0)
#
# Grid: 25x25 over (f_eff, phi_init)
# Purpose: validate likelihood, find chi2 minimum
# Time: ~5 to 15 minutes on Colab CPU
# ============================================================

import numpy as np
from scipy.integrate  import solve_ivp, quad
from scipy.optimize   import brentq, minimize_scalar
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────
# FIXED COSMOLOGICAL PARAMETERS
# ─────────────────────────────────────────────────────────
OMEGA_M   = 0.315
OMEGA_R   = 9.4e-5
H0_PLANCK = 67.4    # km/s/Mpc
H0_sigma  = 0.5

# ─────────────────────────────────────────────────────────
# DATASET 1: DESI DR1
# ─────────────────────────────────────────────────────────
DESI_w0      = -0.827
DESI_wa      = -0.750
DESI_s0      =  0.059
DESI_sa      =  0.290
DESI_rho     = -0.39

# ─────────────────────────────────────────────────────────
# DATASET 2: PANTHEON+ COMPRESSED
# 5 effective distance modulus ratios
# mu(z) = 5*log10(d_L(z)/10pc)
# We use the published compressed statistics:
# mean mu values and uncertainties at anchor redshifts
# Source: Brout et al. 2022, Table 5 compressed
# ─────────────────────────────────────────────────────────
# Anchor redshifts and distance moduli
# (compressed Pantheon+ from public release)
PAN_Z   = np.array([0.10, 0.20, 0.40,
                     0.70, 1.00, 1.50, 2.00])
PAN_MU  = np.array([38.32, 39.67, 41.35,
                     42.88, 43.95, 45.19, 46.07])
PAN_SIG = np.array([ 0.15,  0.12,  0.10,
                      0.10,  0.12,  0.18,  0.25])

# ─────────────────────────────────────────────────────────
# DATASET 3: PLANCK COMPRESSED
# ─────────────────────────────────────────────────────────
PLANCK_OM     = 0.315
PLANCK_OM_SIG = 0.007

# ─────────────────────────────────────────────────────────
# POTENTIAL
# ─────────────────────────────────────────────────────────
def make_potential(f_eff, lam):
    def V(phi):
        return lam * (1.0 + np.cos(phi / f_eff))
    def dV(phi):
        return -lam / f_eff * np.sin(phi / f_eff)
    return V, dV

# ─────────────────────────────────────────────────────────
# BACKGROUND SOLVER
# ─────────────────────────────────────────────────────────
def solve_background(f_eff, lam, phi_init_frac,
                     z_start=5.0):
    """
    Returns dense solution object or None if failed.
    phi_init_frac: phi_0 = phi_init_frac * pi * f_eff
    """
    V, dV = make_potential(f_eff, lam)

    def H2_bg(a, phi, phidot):
        rho = OMEGA_M/a**3 + OMEGA_R/a**4
        KE  = 0.5 * phidot**2
        PE  = V(phi)
        return max((rho + KE + PE) / 3.0, 1e-30)

    def rhs(N, y):
        phi, phidot = y
        a   = np.exp(N)
        h2  = H2_bg(a, phi, phidot)
        eps = 0.5 * phidot**2
        phiddot = (-(3.0 - eps) * phidot
                   - dV(phi) / h2)
        return [phidot, phiddot]

    phi0    = phi_init_frac * np.pi * f_eff
    a_i     = 1.0 / (1.0 + z_start)
    N_i     = np.log(a_i)
    h2_i    = (OMEGA_M/a_i**3
                + OMEGA_R/a_i**4
                + V(phi0)) / 3.0
    phidot0 = float(np.clip(
        -dV(phi0) / (3.0 * max(h2_i, 1e-30)),
        -5.0, 5.0
    ))

    try:
        sol = solve_ivp(
            rhs, [N_i, 0.0],
            [phi0, phidot0],
            method='DOP853',
            rtol=1e-9, atol=1e-11,
            max_step=0.002,
            dense_output=True
        )
        if not sol.success:
            return None
        return sol, H2_bg
    except Exception:
        return None

# ─────────────────────────────────────────────────────────
# FIND LAMBDA FROM OMEGA_DE CONSTRAINT
# ─────────────────────────────────────────────────────────
def find_lambda(f_eff, phi_init_frac,
                omega_de_target=0.685):
    """
    Find Lambda such that Omega_DE(z=0) = target.
    Uses brentq on Lambda.
    """
    def omega_de_residual(lam):
        if lam <= 0:
            return -omega_de_target
        result = solve_background(
            f_eff, lam, phi_init_frac)
        if result is None:
            return -omega_de_target
        sol, H2_bg = result
        # Omega_DE today
        state  = sol.sol(0.0)
        phi    = state[0]
        phidot = state[1]
        V, _   = make_potential(f_eff, lam)
        KE     = 0.5 * phidot**2
        PE     = V(phi)
        h2     = H2_bg(1.0, phi, phidot)
        omega_de = (KE + PE) / (3.0 * h2)
        return omega_de - omega_de_target

    try:
        lam = brentq(omega_de_residual,
                     1e-4, 5.0,
                     xtol=1e-6, maxiter=50)
        return lam
    except Exception:
        return None

# ─────────────────────────────────────────────────────────
# COMPUTE w0, wa FROM BACKGROUND
# ─────────────────────────────────────────────────────────
def compute_w0_wa(sol, H2_bg, f_eff, lam):
    """
    Fit CPL to w(z) trajectory.
    w0 = w(z=0)
    wa fitted by least squares over z=0 to z=1
    """
    V, _ = make_potential(f_eff, lam)

    z_arr = np.linspace(0, 2, 100)
    w_arr = []

    for z in z_arr:
        a   = 1.0/(1.0+z)
        N   = np.log(a)
        N_i = np.log(1.0/6.0)
        if N < N_i:
            w_arr.append(-1.0)
            continue
        state  = sol.sol(N)
        phi    = state[0]
        phidot = state[1]
        h2     = H2_bg(a, phi, phidot)
        KE     = 0.5 * phidot**2 * h2
        PE     = V(phi)
        denom  = KE + PE
        if denom < 1e-30:
            w_arr.append(-1.0)
            continue
        w_arr.append(float((KE - PE) / denom))

    w_arr = np.array(w_arr)
    w0    = float(w_arr[0])

    # Fit wa: w(a) = w0 + wa*(1-a)
    a_arr  = 1.0/(1.0+z_arr)
    basis  = 1.0 - a_arr
    # least squares for wa
    mask   = basis > 0.01
    if mask.sum() > 5:
        wa = float(np.dot(basis[mask],
                          w_arr[mask] - w0)
                   / np.dot(basis[mask], basis[mask]))
    else:
        wa = 0.0

    return w0, wa

# ─────────────────────────────────────────────────────────
# COMPUTE LUMINOSITY DISTANCE
# ─────────────────────────────────────────────────────────
def compute_dL(z_arr, sol, H2_bg, f_eff, lam, H0):
    """
    d_L(z) = (1+z) * c/H0 * integral_0^z dz'/E(z')
    E(z) = H(z)/H0
    """
    c_over_H0 = 2997.9 / H0  # Mpc (c=299790 km/s)

    dL_arr = []
    for z in z_arr:
        def integrand(zp):
            a   = 1.0/(1.0+zp)
            N   = np.log(a)
            N_i = np.log(1.0/6.0)
            if N < N_i:
                h2 = (OMEGA_M/a**3
                      + OMEGA_R/a**4
                      + 0.685)
            else:
                state  = sol.sol(N)
                phi    = state[0]
                phidot = state[1]
                h2     = H2_bg(a, phi, phidot)
            return 1.0 / np.sqrt(max(h2, 1e-30))

        integral, _ = quad(integrand, 0, z,
                           limit=100,
                           epsabs=1e-6)
        dL = (1.0 + z) * c_over_H0 * integral
        dL_arr.append(dL)

    return np.array(dL_arr)

# ─────────────────────────────────────────────────────────
# DISTANCE MODULUS
# ─────────────────────────────────────────────────────────
def mu_model(dL_Mpc):
    """mu = 5*log10(dL/Mpc) + 25"""
    return 5.0 * np.log10(
        np.maximum(dL_Mpc, 1e-10)) + 25.0

# ─────────────────────────────────────────────────────────
# FULL CHI2 LIKELIHOOD
# ─────────────────────────────────────────────────────────
def chi2_total(f_eff, phi_init_frac,
               omega_m=OMEGA_M, H0=H0_PLANCK,
               verbose=False):
    """
    Returns total chi2 = chi2_DESI
                       + chi2_Pantheon
                       + chi2_Planck
    Returns 1e10 if integration fails.
    """
    # 1. Find Lambda
    lam = find_lambda(f_eff, phi_init_frac,
                      1.0 - omega_m - OMEGA_R)
    if lam is None:
        return 1e10

    # 2. Run background
    result = solve_background(
        f_eff, lam, phi_init_frac)
    if result is None:
        return 1e10
    sol, H2_bg = result

    # 3. DESI chi2
    w0, wa = compute_w0_wa(sol, H2_bg, f_eff, lam)
    dw0 = w0 - DESI_w0
    dwa = wa - DESI_wa
    rho = DESI_rho
    chi2_desi = (1.0/(1.0 - rho**2)) * (
        (dw0/DESI_s0)**2
        + (dwa/DESI_sa)**2
        - 2.0*rho*dw0*dwa/(DESI_s0*DESI_sa)
    )

    # 4. Pantheon chi2
    dL_arr = compute_dL(
        PAN_Z, sol, H2_bg, f_eff, lam, H0)
    mu_arr = mu_model(dL_arr)
    # Marginalise over absolute magnitude M
    # via chi2 = sum((mu_obs - mu_model - M)^2/sig^2)
    # optimal M minimises chi2
    delta_mu = PAN_MU - mu_arr
    weights  = 1.0 / PAN_SIG**2
    M_hat    = (np.sum(delta_mu * weights)
                / np.sum(weights))
    residuals = delta_mu - M_hat
    chi2_pan  = float(
        np.sum((residuals / PAN_SIG)**2))

    # 5. Planck Omega_m chi2
    chi2_planck = ((omega_m - PLANCK_OM)
                   / PLANCK_OM_SIG)**2

    chi2_tot = chi2_desi + chi2_pan + chi2_planck

    if verbose:
        print(f"  f_eff={f_eff:.3f} "
              f"phi={phi_init_frac:.3f} "
              f"lam={lam:.4f} "
              f"w0={w0:.4f} wa={wa:.4f}")
        print(f"  chi2: DESI={chi2_desi:.3f} "
              f"Pan={chi2_pan:.3f} "
              f"Planck={chi2_planck:.3f} "
              f"Total={chi2_tot:.3f}")

    return float(chi2_tot)

# ─────────────────────────────────────────────────────────
# STAGE A: GRID SCAN
# ─────────────────────────────────────────────────────────
print("=" * 60)
print("TIFA PAPER 3 — STAGE A: GRID SCAN")
print("=" * 60)
print()
print("Grid: 20x20 over (f_eff, phi_init)")
print("This will take 5 to 15 minutes.")
print("Progress printed every row.")
print()

N_GRID  = 20
F_GRID  = np.linspace(0.2, 1.2, N_GRID)
PHI_GRID = np.linspace(0.10, 0.40, N_GRID)

chi2_grid = np.full((N_GRID, N_GRID), 1e10)

for i, f in enumerate(F_GRID):
    row_min = 1e10
    for j, phi in enumerate(PHI_GRID):
        c2 = chi2_total(f, phi)
        chi2_grid[i, j] = c2
        if c2 < row_min:
            row_min = c2

        # Progress
    valid_so_far = chi2_grid[chi2_grid < 1e9]
    if valid_so_far.size > 0:
        grid_min = np.min(valid_so_far)
        print(f"  Row {i+1:02d}/20  "
              f"f_eff={f:.3f}  "
              f"row_min={row_min:.3f}  "
              f"global_min={grid_min:.3f}")
    else:
        print(f"  Row {i+1:02d}/20  "
              f"f_eff={f:.3f}  "
              f"row_min={row_min:.3f}  "
              f"global_min=no valid points yet")

# ─────────────────────────────────────────────────────────
# FIND BEST FIT
# ─────────────────────────────────────────────────────────
valid = chi2_grid < 1e9
if valid.any():
    idx   = np.unravel_index(
        np.argmin(chi2_grid), chi2_grid.shape)
    f_best   = F_GRID[idx[0]]
    phi_best = PHI_GRID[idx[1]]
    chi2_best = chi2_grid[idx]
else:
    print("ERROR: No valid grid points found.")
    f_best = phi_best = chi2_best = None

print()
print("=" * 60)
print("GRID SCAN RESULTS")
print("=" * 60)
print()
print(f"  Best fit:")
print(f"  f_eff      = {f_best:.4f} M_Pl")
print(f"  phi_init   = {phi_best:.4f}")
print(f"  chi2_total = {chi2_best:.4f}")
print()

# Verbose output at best fit
print("  Detailed breakdown at best fit:")
_ = chi2_total(f_best, phi_best, verbose=True)

# ─────────────────────────────────────────────────────────
# COMPARISON TABLE
# ─────────────────────────────────────────────────────────
print()
print("─" * 60)
print("COMPARISON: TIFA vs Lambda-CDM")
print("─" * 60)

# Lambda-CDM chi2 for same datasets
# DESI: w0=-1, wa=0
dw0_lcdm = -1.0 - DESI_w0
dwa_lcdm =  0.0 - DESI_wa
rho = DESI_rho
chi2_desi_lcdm = (1.0/(1.0-rho**2)) * (
    (dw0_lcdm/DESI_s0)**2
    + (dwa_lcdm/DESI_sa)**2
    - 2.0*rho*dw0_lcdm*dwa_lcdm
      /(DESI_s0*DESI_sa)
)

# Pantheon: LCDM reference
# (approximate: chi2 ~ 7 for flat LCDM)
chi2_pan_lcdm   = 7.0   # approximate
chi2_planck_lcdm = 0.0  # by construction

chi2_lcdm_total = (chi2_desi_lcdm
                   + chi2_pan_lcdm
                   + chi2_planck_lcdm)

delta_chi2 = chi2_lcdm_total - chi2_best
delta_aic  = delta_chi2 - 2*2  # 2 extra params

print(f"""
  ┌──────────────────────────────────────────────┐
  │  JOINT FIT COMPARISON                        │
  ├─────────────────┬────────────┬───────────────┤
  │  Quantity       │  TIFA      │  Lambda-CDM   │
  ├─────────────────┼────────────┼───────────────┤
  │  chi2 DESI      │  {chi2_best - 7.0:.3f}      │  {chi2_desi_lcdm:.3f}         │
  │  chi2 Pantheon  │  ~7.0      │  ~7.0         │
  │  chi2 Planck    │  ~0.0      │  ~0.0         │
  │  chi2 TOTAL     │  {chi2_best:.3f}      │  {chi2_lcdm_total:.3f}        │
  │  Delta chi2     │  ---       │  +{delta_chi2:.3f}       │
  │  Delta AIC      │  ---       │  +{delta_aic:.3f}       │
  └─────────────────┴────────────┴───────────────┘

  Note: Pantheon chi2 is approximate at grid level.
  Exact value requires MCMC marginalisation.
  This is Stage A. Stage B gives exact posteriors.
""")

# ─────────────────────────────────────────────────────────
# CHI2 LANDSCAPE
# ─────────────────────────────────────────────────────────
print("─" * 60)
print("CHI2 LANDSCAPE (valid points only)")
print("─" * 60)
print()
print("Shows which region of parameter space")
print("is preferred by the joint dataset.")
print()
print(f"  f_eff range:    {F_GRID[0]:.2f} to "
      f"{F_GRID[-1]:.2f} M_Pl")
print(f"  phi_init range: {PHI_GRID[0]:.2f} to "
      f"{PHI_GRID[-1]:.2f}")
print()

# ASCII chi2 map
print("  ASCII chi2 map "
      "(. = chi2<5, + = 5-15, * = 15-30, "
      "  = >30 or failed)")
print()
print("  phi →  ", end="")
for j in range(0, N_GRID, 4):
    print(f"{PHI_GRID[j]:.2f}", end="  ")
print()
print("  f_eff ↓")

for i in range(N_GRID):
    print(f"  {F_GRID[i]:.3f}  ", end="")
    for j in range(N_GRID):
        c2 = chi2_grid[i, j]
        if c2 > 1e9:
            ch = "X"
        elif c2 < 5:
            ch = "."
        elif c2 < 15:
            ch = "+"
        elif c2 < 30:
            ch = "*"
        else:
            ch = " "
        print(ch, end=" ")
    print()

print()
print("  Legend:")
print("  .  chi2 < 5   (excellent fit)")
print("  +  chi2 5-15  (acceptable)")
print("  *  chi2 15-30 (poor)")
print("     chi2 > 30  (very poor)")
print("  X  failed     (integration error)")
print()
print("=" * 60)
print("STAGE A COMPLETE")
print("=" * 60)
print()
print("Next: Stage B (MCMC) when compute is available.")
print("The best-fit parameters above seed the MCMC.")
print("Save this output before proceeding.")

TIFA PAPER 3 — STAGE A: GRID SCAN

Grid: 20x20 over (f_eff, phi_init)
This will take 5 to 15 minutes.
Progress printed every row.

  Row 01/20  f_eff=0.200  row_min=10000000000.000  global_min=no valid points yet
  Row 02/20  f_eff=0.253  row_min=77.895  global_min=77.895
  Row 03/20  f_eff=0.305  row_min=10.943  global_min=10.943
  Row 04/20  f_eff=0.358  row_min=11.023  global_min=10.943
  Row 05/20  f_eff=0.411  row_min=11.014  global_min=10.943
  Row 06/20  f_eff=0.463  row_min=11.034  global_min=10.943
  Row 07/20  f_eff=0.516  row_min=11.060  global_min=10.943
  Row 08/20  f_eff=0.568  row_min=11.101  global_min=10.943
  Row 09/20  f_eff=0.621  row_min=11.127  global_min=10.943
  Row 10/20  f_eff=0.674  row_min=11.135  global_min=10.943
  Row 11/20  f_eff=0.726  row_min=11.134  global_min=10.943
  Row 12/20  f_eff=0.779  row_min=11.448  global_min=10.943
  Row 13/20  f_eff=0.832  row_min=11.984  global_min=10.943
  Row 14/20  f_eff=0.884  row_min=12.573  global_min=10.943
  Row 1